In [1]:
# Package version audit — run this at the start of every session
# to confirm the environment matches what the study was built on

import sys
import importlib

packages = {
    "opendssdirect": "opendssdirect",
    "numpy":         "numpy",
    "pandas":        "pandas",
    "matplotlib":    "matplotlib",
    "jupyterlab":    "jupyterlab",
}

print(f"Python : {sys.version}")
print("-" * 48)
for label, mod in packages.items():
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "version attr not found")
        print(f"{label:<20} {ver}")
    except ImportError as e:
        print(f"{label:<20} *** NOT INSTALLED: {e}")

Python : 3.13.12 (main, Feb  4 2026, 15:06:39) [GCC 15.2.0]
------------------------------------------------
opendssdirect        0.9.4
numpy                2.4.6
pandas               3.0.3
matplotlib           3.10.9
jupyterlab           4.5.8


In [4]:
import opendssdirect as dss
from pathlib import Path

# Start the OpenDSS engine
dss.Basic.Start(0)
print(f"OpenDSS engine started   : {dss.Basic.COMErrorResults()!r}")
print(f"OpenDSS version          : {dss.__version__}")
print(f"DSS engine version       : {dss.Basic.Version()}")

# Run a minimal one-line network to confirm the solver is operational
test_script = """
Clear
New Circuit.TestSource basekv=11 pu=1.0 angle=0 frequency=50 mvasc3=500 mvasc1=450
New Line.TestLine bus1=sourcebus bus2=testbus length=1 units=km r1=0.342 x1=0.310
New Load.TestLoad bus1=testbus phases=3 kv=11 kw=100 pf=0.9
Set Mode=Snapshot
Solve
"""

dss.run_command(test_script)

if dss.Solution.Converged():
    print("\nTest circuit solve       : CONVERGED ✓")
    print(f"Active buses             : {dss.Circuit.AllBusNames()}")
    print(f"Total load (kW)          : {-dss.Circuit.TotalPower()[0]:.2f}")
else:
    print("\nTest circuit solve       : FAILED — check OpenDSS installation")

OpenDSS engine started   : True
OpenDSS version          : 0.9.4
DSS engine version       : DSS C-API Library version 0.14.5 revision 87d85c2622c8281b92255335bc7c09b11191b21d based on OpenDSS SVN 3723 [FPC 3.2.2] (64-bit build) MVMULT INCREMENTAL_Y CONTEXT_API PM 20240329033721; License Status: Open 
DSS-Python version: 0.15.7
OpenDSSDirect.py version: 0.9.4

Test circuit solve       : CONVERGED ✓
Active buses             : ['sourcebus', 'testbus']
Total load (kW)          : -0.00


/tmp/ipykernel_3727616/3146685273.py:20: DeprecationWarning: run_command is deprecated (use Command, Commands or the callable shortcut), see https://github.com/dss-extensions/OpenDSSDirect.py/issues/70
  dss.run_command(test_script)


In [7]:
from pathlib import Path

#project_root = Path.home() / "Desktop" / "Substation + Automation" / "GitProjectsX3" / "fuse-recloser_coordination"
project_root = Path.home() / "Desktop" / "Substation +  Automation" / "GitProjectsX3" / "fuse-recloser_coordination"

required = [
    "dss/master.dss",
    "dss/network.dss",
    "dss/loads.dss",
    "dss/sseg.dss",
    "dss/protection.dss",
    "scripts/run_study.py",
    "notebooks",
    "outputs/plots",
    "outputs/reports",
    "data",
    "README.md",
    ".gitignore",
]

print(f"Project root: {project_root}\n")
print(f"{'Path':<40} {'Status'}")
print("-" * 52)

all_present = True
for item in required:
    full_path = project_root / item
    exists = full_path.exists()
    status = "✓  exists" if exists else "✗  MISSING"
    if not exists:
        all_present = False
    print(f"{item:<40} {status}")

print()
print("All required files present ✓" if all_present else "WARNING: some files are missing — re-run setup.")

Project root: /home/hillary/Desktop/Substation +  Automation/GitProjectsX3/fuse-recloser_coordination

Path                                     Status
----------------------------------------------------
dss/master.dss                           ✓  exists
dss/network.dss                          ✓  exists
dss/loads.dss                            ✓  exists
dss/sseg.dss                             ✓  exists
dss/protection.dss                       ✓  exists
scripts/run_study.py                     ✓  exists
notebooks                                ✓  exists
outputs/plots                            ✓  exists
outputs/reports                          ✓  exists
data                                     ✓  exists
README.md                                ✓  exists
.gitignore                               ✓  exists

All required files present ✓


In [8]:
# ============================================================================
# OpenDSS Execution Model — Engineering Explanation
# This cell documents the paradigm before any network is built.
# ============================================================================

explanation = """
OpenDSS EXECUTION MODEL
========================

OpenDSS is not an interactive GUI tool on Linux — it is a text-driven power
system solver accessed through a Python API. The execution model has three
distinct layers:

1. DSS SCRIPT LAYER (.dss files)
   The network is defined entirely in DSS-language text files. Each element
   (source, line, load, generator, fuse, recloser) is declared as a DSS
   command string. These files are plain text — editable in any text editor.
   The DSS language is case-insensitive and property-order-independent.

2. PYTHON ORCHESTRATION LAYER (run_study.py)
   Python does not define the network — it controls the solver. It tells
   OpenDSS which files to load (via dss.run_command('Redirect master.dss')),
   what study mode to run (Faultstudy, Snapshot, Daily, etc.), and then
   extracts results from OpenDSS memory via the opendssdirect API.
   Think of Python as the study controller and OpenDSS as the calculation
   engine. Results come back as Python lists and floats via API calls like
   dss.Bus.VMagAngle() and dss.Faults.CurrentMagnitudes().

3. RESULT EXTRACTION LAYER
   After each solve, results live inside the OpenDSS engine memory. Python
   reads them out through getter functions. There is no automatic output file
   unless you explicitly write one. This means every result must be
   deliberately extracted and stored — which is actually an advantage because
   you only extract what you need for the study.

EXECUTION SEQUENCE for this study:
   Python opens master.dss → master.dss redirects network.dss, loads.dss,
   protection.dss in order → OpenDSS builds the network object model →
   Python calls dss.Solution.Solve() → Python reads fault currents at each
   bus via the API → NumPy/Pandas process results → Matplotlib plots TCCs.

The critical insight: DSS files define WHAT the network is.
                     Python defines WHAT to do with it and WHAT to report.
"""

print(explanation)


OpenDSS EXECUTION MODEL

OpenDSS is not an interactive GUI tool on Linux — it is a text-driven power
system solver accessed through a Python API. The execution model has three
distinct layers:

1. DSS SCRIPT LAYER (.dss files)
   The network is defined entirely in DSS-language text files. Each element
   (source, line, load, generator, fuse, recloser) is declared as a DSS
   command string. These files are plain text — editable in any text editor.
   The DSS language is case-insensitive and property-order-independent.

2. PYTHON ORCHESTRATION LAYER (run_study.py)
   Python does not define the network — it controls the solver. It tells
   OpenDSS which files to load (via dss.run_command('Redirect master.dss')),
   what study mode to run (Faultstudy, Snapshot, Daily, etc.), and then
   extracts results from OpenDSS memory via the opendssdirect API.
   Think of Python as the study controller and OpenDSS as the calculation
   engine. Results come back as Python lists and floats via API ca

In [9]:
import subprocess
from pathlib import Path

project_root = Path.home() / "Desktop" / "Substation +  Automation" / "GitProjectsX3" / "fuse-recloser_coordination"

# Check git status
result = subprocess.run(
    ["git", "status", "--short"],
    cwd=project_root,
    capture_output=True,
    text=True
)
print("Git status:\n")
print(result.stdout if result.stdout else "(working tree clean)")

# Show what will be committed
result2 = subprocess.run(
    ["git", "log", "--oneline", "-5"],
    cwd=project_root,
    capture_output=True,
    text=True
)
print("\nRecent commits:")
print(result2.stdout if result2.stdout else "(no commits yet — ready for initial commit)")

Git status:

?? .gitignore
?? README.md
?? Untitled.ipynb
?? dss/
?? notebooks/
?? scripts/
?? section1_setup_verification.ipynb


Recent commits:
(no commits yet — ready for initial commit)
